# 01 — Exploratory Data Analysis

Political sentiment corpus (English + Bangla). This notebook builds/loads the dataset and explores class balance, language mix, text length, party/topic coverage, and negation/sarcasm phenomena.

> Run `make data` first, or execute the build cell below.

In [ ]:
import sys; sys.path.insert(0, '../src')
import pandas as pd, matplotlib.pyplot as plt
from political_sentiment.data.io import load_frame
pd.set_option('display.max_colwidth', 80)

In [ ]:
# (Re)build the dataset if needed
import subprocess, os
if not os.path.exists('../data/processed/train.csv') and not os.path.exists('../data/processed/train.parquet'):
    subprocess.run(['python','-m','political_sentiment.data.build_dataset','--config','../configs/base.yaml'], cwd='..')

In [ ]:
train = load_frame('../data/processed','train')
val   = load_frame('../data/processed','val')
test  = load_frame('../data/processed','test')
print(train.shape, val.shape, test.shape)
train.head()

## Class balance

In [ ]:
ax = train['label'].value_counts().plot(kind='bar', title='Label distribution (train)')
ax.set_ylabel('count'); plt.tight_layout(); plt.show()

## Language mix and per-language class balance

In [ ]:
print(train['language'].value_counts(normalize=True).round(3))
pd.crosstab(train['language'], train['label'], normalize='index').round(3)

## Text length distribution

In [ ]:
train['n_chars'] = train['text'].str.len()
train['n_words'] = train['text'].str.split().apply(len)
train[['n_chars','n_words']].describe().round(1)

In [ ]:
train.groupby('language')['n_words'].plot(kind='hist', alpha=0.5, legend=True, bins=20, title='Words per post by language'); plt.tight_layout(); plt.show()

## Party and topic coverage (used only for bias auditing, not as features)

In [ ]:
pd.crosstab(train['party'], train['label'])

In [ ]:
pd.crosstab(train['topic'], train['label'])

## Negation & sarcasm examples
The generator injects negation, contrastive and sarcastic constructions that make the task realistically hard.

In [ ]:
mask = train['text'].str.contains('not |মোটেও|নয়|Oh |বাহ', regex=True)
train[mask][['language','label','text']].head(10)

### Takeaways
- Three classes are near-balanced by construction; ~6% label noise sets a realistic accuracy ceiling (~0.94).
- English and Bangla are ~50/50; both carry all three classes.
- Negation/contrast/sarcasm are present in a meaningful fraction — these are the hard cases the deep models should help with.